# Day 12 · NumPy 进阶

**引入:`*` 是对位乘,`@` 是行乘列**

抽问上节:`np.array([1,2,3]) * np.array([4,5,6])` 结果是什么?(`[4, 10, 18]`,逐元素)。再问:真正线性代数的“矩阵乘法”呢?引出**`@` 运算符**。现场演示 1000x1000 矩阵乘法:Python 三重循环几十秒 vs `np.dot` 毫秒级 —— **线性代数是数据科学的数学底座**。

**导入 NumPy**

每个 cell 开头重新 `import numpy as np`,确保可以单独 shift+enter 运行,也是 Day11 培养的固定习惯。

In [ ]:
# 导入 NumPy 社区标准别名 np
import numpy as np

**逐元素乘法 * vs 矩阵乘法 @**

**`*`** = 对应位置相乘(逐元素),要求 shape 一致或可广播。**`@`** = 真正的线性代数矩阵乘法(行乘列),`A @ B` 的语义是 **A 的行** 点乘 **B 列**,要求 A 的列数 = B 的行数。矩阵乘法**不满足交换律**,`A @ B` ≠ `B @ A`。

In [ ]:
import numpy as np

# 创建两个 2x2 矩阵用于演示
A = np.array([[1, 2],
              [3, 4]])
B = np.array([[5, 6],
              [7, 8]])

# 逐元素乘法:对应位置相乘
print(A * B)
# [[ 5 12]
#  [21 32]]  ← 1*5,2*6,3*7,4*8

# 矩阵乘法 A @ B:行乘列
print(A @ B)
# [[19 22]
#  [43 50]]  ← (1*5+2*7)=19,(1*6+2*8)=22,...

# np.dot 等价于 @,老项目常见写法
print(np.dot(A, B))

**转置 .T**

`.T` 属性把行变列、列变行,`m x n` → `n x m`。元素 `A[i][j]` 变为 `A.T[j][i]`。转置在矩阵运算中极其常见,例如计算协方差、最小二乘。

In [ ]:
import numpy as np

# 2 行 3 列矩阵
M = np.array([[1, 2, 3],
              [4, 5, 6]])
print(M.shape)        # (2, 3)

# 转置后变成 3 行 2 列
print(M.T)
print(M.T.shape)      # (3, 2)

**行列式与逆矩阵 —— np.linalg.det / np.linalg.inv**

`np.linalg.det` 求**行列式**(方阵特有),非零才可逆。`np.linalg.inv` 求**逆矩阵**,满足 `A @ inv(A) = I`(单位矩阵)。数值计算中推荐用 `np.linalg.solve` 求方程(更稳定),逆矩阵主要用于演示和验证理论。

In [ ]:
import numpy as np

A = np.array([[1, 2],
              [3, 4]])

# 行列式:ad-bc = 1*4 - 2*3 = -2
print(np.linalg.det(A))   # -2.0(浮点数,可能有微小误差)

# 逆矩阵
inv_A = np.linalg.inv(A)
print(inv_A)

# 验证:A @ inv(A) ≈ 单位矩阵 I(对角 1,其余 0)
print(np.round(A @ inv_A, 8))

**聚合函数 + 轴 axis 概念**

对整个数组聚合得到标量;指定 **axis** 沿该维度压缩。**`axis=0` 跨行(压扁行)**,结果长度 = 列数;**`axis=1` 跨列(压扁列)**,结果长度 = 行数。记忆:“axis=0 竖着跨,按列下来”。

In [ ]:
import numpy as np

# 2 行 3 列矩阵
M = np.array([[1, 2, 3],
              [4, 5, 6]])

# 不指定 axis → 整个数组的统计
print(M.sum())            # 21
print(M.mean())           # 3.5
print(M.std())            # 标准差 ≈ 1.7078

# axis=0:沿行方向压扁,每列一个数
print(M.sum(axis=0))      # [5 7 9]  ← 每列求和

# axis=1:沿列方向压扁,每行一个数
print(M.sum(axis=1))      # [ 6 15]  ← 每行求和

**argmax 扁平化索引 + unravel_index**

`np.argmax` 把数组**按行展平**(flatten)后返回最大值的**扁平索引**,不告诉你是第几行第几列。要还原为二维坐标,用 `np.unravel_index(index, shape)`,它能根据形状反算出行列位置。

In [ ]:
import numpy as np

M = np.array([[1, 9, 3],
              [4, 5, 6]])

# argmax 默认把数组展平后返回最大值的索引
# 展平后: [1, 9, 3, 4, 5, 6],最大值 9 在位置 1
flat_idx = M.argmax()
print(flat_idx)           # 1

# unravel_index 根据形状 (2, 3) 反算行列坐标
row, col = np.unravel_index(flat_idx, M.shape)
print((row, col))         # (0, 1) —— 第 0 行第 1 列

# 直接在二维上用 argmax(axis=...) 更方便
# 但扁平索引 + unravel_index 是处理高维数组时的通用方法

**随机种子 —— 可复现实验**

机器学习、模拟实验需要结果**可复现**,调用 `np.random.seed(整数)` 后,随机数生成器重置,每次运行得到**相同**的随机数据。**常用写法 `np.random.seed(42)`**,42 是编程文化梗(《银河系漫游指南》“生命、宇宙与一切的答案”)。

In [ ]:
import numpy as np

# 设下种子,之后每次运行得到相同随机数
np.random.seed(42)

# 生成 3 行 2 列 [0,1) 均匀分布随机矩阵
print(np.random.rand(3, 2))

# 再次设同样种子,生成与上方完全一致的随机数
np.random.seed(42)
print(np.random.rand(3, 2))  # 与上一块输出完全相同

**正态分布随机数 —— np.random.normal**

`np.random.normal(loc, scale, size)` 生成**正态分布**(高斯分布)随机数。`loc` 均值、`scale` 标准差、`size` 输出形状。约 68% 数据落在 (均值 ± 标准差) 区间内。

In [ ]:
import numpy as np

# 均值 0、标准差 1 的标准正态分布 N(0,1),5 个样本
samples = np.random.normal(loc=0.0, scale=1.0, size=5)
print(samples)

# 均值 100、标准差 15(类似 IQ 分数分布),10000 个样本
iq = np.random.normal(loc=100, scale=15, size=10000)
print(iq.mean())     # ≈ 100,样本量越大越接近理论均值
print(iq.std())      # ≈ 15

**均匀分布随机整数 —— np.random.randint**

`np.random.randint(low, high, size)` 生成 **[low, high) 区间**左闭右开的随机整数。常用于模拟掷骰子、洗牌、随机抽样等场景。注意 `high` 取不到。

In [ ]:
import numpy as np

# 模拟掷 10 次六面骰子,取值 1~6(7 取不到)
dice = np.random.randint(1, 7, size=10)
print(dice)           # 例如 [3 6 2 5 1 4 2 3 1 4]

# 生成 3 行 4 列 [0, 10) 整数矩阵
mat = np.random.randint(0, 10, size=(3, 4))
print(mat)

**保存与加载 —— np.save / np.load(二进制 .npy)**

`np.save(file, arr)` 把**单个数组**保存为 `.npy` 二进制文件,保留 dtype / shape 完整信息。`np.load(file)` 加载还原。二进制格式**读写快、体积小**,是 NumPy 原生持久化首选。

In [ ]:
import numpy as np

# 硬编码示例数据:2 行 3 列矩阵
data = np.array([[10, 20, 30],
                 [40, 50, 60]])

# 保存为 .npy 二进制文件(文件名末尾的 .npy 可省略,自动补上)
np.save('my_array.npy', data)

# 加载回来,dtype / shape 完美还原
loaded = np.load('my_array.npy')
print(loaded)
print(loaded.dtype, loaded.shape)

**保存与加载 —— np.savetxt / np.loadtxt(文本 .csv)**

`np.savetxt` / `np.loadtxt` 读写**文本格式**(csv / txt),人类可用 Excel 打开。必须指定 `delimiter` 分隔符,`header` 表头,`comments=''` 禁用默认 `#` 前缀,否则首行被注释掉。

In [ ]:
import numpy as np

# 硬编码示例数据
data = np.array([[1.1, 2.2, 3.3],
                 [4.4, 5.5, 6.6]])

# 保存为 csv,delimiter 分隔符用逗号
# comments='' 避免 header 行被加 # 前缀(被注释就废了)
np.savetxt('my_array.csv', data, delimiter=',',
           header='col1,col2,col3', comments='')

# 加载 csv,同样要指定 delimiter,skiprows=1 跳过表头行
loaded = np.loadtxt('my_array.csv', delimiter=',', skiprows=1)
print(loaded)
print(loaded.dtype)   # float64,文本文件统一读成浮点

**今日小结**

- **`*` 逐元素乘 vs `@` 矩阵乘**,`np.dot` 等价于 `@`
- **`.T` 转置**,行列互换
- **`np.linalg.det` 行列式**、**`np.linalg.inv` 逆矩阵**
- **axis=0 跨行聚合**、**axis=1 跨列聚合**
- **argmax 扁平索引 + unravel_index 反算坐标**
- **`np.random.seed(42)` 可复现种子**
- **`np.random.normal` 正态分布**、**`np.random.randint` 均匀随机整数**
- **`np.save / np.load` 二进制持久化**
- **`np.savetxt / np.loadtxt` 文本 csv 持久化**

**更多练习**

- 当堂练:course/lesson12/in_class/practice01-06.py
- 课后作业:course/lesson12/homework/task01-03.py